# OmicsMind Example


This notebook implements the OmicsMind pipeline for :
- `rna.cct`
- `rppa.cct`
- `scna.cct`
- `methylation.cct`

Workflow:
1. Load & auto-clean each modality (auto delimiter, auto transpose if needed).
2. Align samples across modalities and z-score normalize features.
3. Block-wise masked-modality training:
   - Each modality has a VAE to produce latent tokens.
   - Latent tokens are fused by a cross-omics Transformer.
   - Randomly mask whole modalities during training.
4. Realism & Biological Reasonableness evaluations.
5. Full imputation and CSV export.

If any file is missing, the notebook will generate synthetic data to keep everything runnable.


In [1]:
!pip -q install pandas numpy scikit-learn tqdm torch

import os, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

print("torch:", torch.__version__)


torch: 2.5.1+cu121


## 1. Load and auto-clean `.cct` tables


In [3]:
def _read_table_auto(path):
    # Automatically infer delimiter: comma / tab / whitespace
    return pd.read_csv(path, sep=None, engine="python")

def load_omics_table(path):
    '''
    Goal: return a float DataFrame with shape (samples × features).
    Auto-fixes common formats:
      - First column is sample_id -> set as index
      - If features are in rows and samples in columns -> transpose
      - Non-numeric -> NaN
    '''
    df = _read_table_auto(path)

    # If first column looks like sample ids (strings), set it as index
    if df.shape[1] > 1 and df.iloc[:, 0].dtype == object:
        df = df.set_index(df.columns[0])

    # Heuristic for transpose:
    # If index mostly strings (feature names) but columns mostly numeric/sample-like
    # and rows > cols, treat as (features × samples) and transpose.
    idx_obj_ratio = np.mean([isinstance(x, str) for x in df.index])
    col_obj_ratio = np.mean([isinstance(x, str) for x in df.columns])
    if idx_obj_ratio > 0.8 and col_obj_ratio < 0.2 and df.shape[0] > df.shape[1]:
        df = df.T

    # Force numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    return df


data_dir = "./data/GBM"   
files = {
    "rna": "rna.cct",
    "rppa": "rppa.cct",
    "scna": "scna.cct",
    "meth": "methylation.cct",
}

modalities = {}
missing_files = []
for k, fn in files.items():
    p = os.path.join(data_dir, fn)
    if os.path.exists(p):
        modalities[k] = load_omics_table(p)
        print(k, "loaded", modalities[k].shape)
    else:
        missing_files.append(p)

if missing_files:
    print("\n[WARN] Missing files, will use synthetic data:", missing_files)


rna loaded (595, 19660)
rppa loaded (595, 158)
scna loaded (595, 24776)
meth loaded (595, 20114)


## 2. If files are missing, generate synthetic data


In [4]:
if missing_files:
    np.random.seed(0)
    n = 80
    modalities = {
        "rna":  pd.DataFrame(np.random.randn(n, 2000), index=[f"S{i:03d}" for i in range(n)]),
        "rppa": pd.DataFrame(np.random.randn(n, 200),  index=[f"S{i:03d}" for i in range(n)]),
        "scna": pd.DataFrame(np.random.randn(n, 500),  index=[f"S{i:03d}" for i in range(n)]),
        "meth": pd.DataFrame(np.random.randn(n, 1000), index=[f"S{i:03d}" for i in range(n)]),
    }
    # Simulate block-wise missingness
    idx = modalities["rna"].index
    mask_rppa = np.random.choice(idx, size=int(0.3*n), replace=False)
    mask_meth = np.random.choice(idx, size=int(0.2*n), replace=False)
    modalities["rppa"].loc[mask_rppa] = np.nan
    modalities["meth"].loc[mask_meth] = np.nan

    print("Synthetic modalities:", {k:v.shape for k,v in modalities.items()})


## 3. Sample alignment + z-score normalization


In [5]:
omics_keys = list(files.keys())
all_samples = sorted(set().union(*[modalities[k].index for k in omics_keys]))

def align(df): 
    return df.reindex(all_samples)

modalities_aligned = {k: align(modalities[k]) for k in omics_keys}

def zscore(df):
    mu = df.mean(axis=0, skipna=True)
    sd = df.std(axis=0, skipna=True).replace(0, 1.0)
    return (df - mu) / sd

modalities_aligned = {k: zscore(v) for k,v in modalities_aligned.items()}

X = {k: v.to_numpy(dtype=np.float32) for k,v in modalities_aligned.items()}
print({k: X[k].shape for k in X})


{'rna': (595, 19660), 'rppa': (595, 158), 'scna': (595, 24776), 'meth': (595, 20114)}


## 4. Dataset for block-wise masked-modality training


In [6]:
class MultiOmicsDataset(Dataset):
    def __init__(self, X_dict, mask_prob=0.3):
        self.mods = list(X_dict.keys())
        self.X = X_dict
        self.n = next(iter(X_dict.values())).shape[0]
        self.mask_prob = mask_prob

    def __len__(self): 
        return self.n

    def __getitem__(self, idx):
        sample, observed, train_mask = {}, {}, {}
        # observed vs true-missing (original NaN)
        for m in self.mods:
            x_m = self.X[m][idx]
            obs_m = not np.all(np.isnan(x_m))
            observed[m] = obs_m
            sample[m] = np.nan_to_num(x_m, nan=0.0)

        # training-time random modality masking
        for m in self.mods:
            if observed[m] and random.random() < self.mask_prob:
                train_mask[m] = 0
            else:
                train_mask[m] = 1

        return sample, observed, train_mask

dataset = MultiOmicsDataset(X, mask_prob=0.3)
loader = DataLoader(dataset, batch_size=16, shuffle=True)


## 5. Modality VAEs + Cross-Omics Transformer + OmicsMind


In [7]:
class ModalityVAE(nn.Module):
    def __init__(self, in_dim, z_dim=64, h_dim=256, dropout=0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_dim, h_dim), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(h_dim, z_dim)
        self.fc_logvar = nn.Linear(h_dim, z_dim)
        self.decoder = nn.Sequential(
            nn.Linear(z_dim, h_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(h_dim, h_dim), nn.ReLU(),
            nn.Linear(h_dim, in_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparam(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparam(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


class CrossOmicsTransformer(nn.Module):
    def __init__(self, z_dim=64, n_modalities=4, n_heads=4, n_layers=2, ff_dim=256, dropout=0.1):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=z_dim, nhead=n_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.pos_emb = nn.Parameter(torch.randn(1, n_modalities, z_dim))

    def forward(self, Z):
        Z = Z + self.pos_emb[:, :Z.size(1)]
        return self.encoder(Z)


class OmicsMind(nn.Module):
    def __init__(self, in_dims, z_dim=64):
        super().__init__()
        self.mods = list(in_dims.keys())
        self.vaes = nn.ModuleDict({m: ModalityVAE(in_dims[m], z_dim=z_dim) for m in self.mods})
        self.trans = CrossOmicsTransformer(z_dim=z_dim, n_modalities=len(self.mods))

    def forward(self, batch_x, train_mask):
        Z_list, mus, logvars, refined_xhat = [], [], [], {}
        for m in self.mods:
            x_hat, mu, lv, z = self.vaes[m](batch_x[m])
            Z_list.append(z.unsqueeze(1))
            mus.append(mu); logvars.append(lv)

        Z = torch.cat(Z_list, dim=1)  # (B,M,z)

        mask_vec = torch.stack([train_mask[m] for m in self.mods], dim=1).unsqueeze(-1)
        Z_masked = Z * mask_vec

        Z_refined = self.trans(Z_masked)

        for i,m in enumerate(self.mods):
            refined_xhat[m] = self.vaes[m].decode(Z_refined[:,i,:])

        return refined_xhat, mus, logvars


def vae_kl(mu, logvar):
    return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)).mean()


def omicsmind_loss(batch_x, refined_xhat, mus, logvars, observed, train_mask, beta=1e-3, only_masked=False):
    rec = 0.0
    for m in batch_x:
        x_true = batch_x[m]
        x_pred = refined_xhat[m]
        obs_w = observed[m].float().unsqueeze(1)  # (B,1)

        if only_masked:
            m_w = (1 - train_mask[m]).float().unsqueeze(1)  # only masked modalities
            w = obs_w * m_w
        else:
            w = obs_w

        rec += (((x_true - x_pred)**2) * w).mean()

    kl = sum(vae_kl(mu, lv) for mu,lv in zip(mus, logvars))
    return rec + beta*kl, rec, kl


## 6. Training


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
in_dims = {m: X[m].shape[1] for m in X}

model = OmicsMind(in_dims, z_dim=64).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

def to_tensor_batch(batch):
    sample, observed, train_mask = batch
    batch_x = {m: torch.tensor(sample[m], device=device) for m in sample}
    observed_t = {m: torch.tensor(observed[m], device=device) for m in observed}
    train_mask_t = {m: torch.tensor(train_mask[m], device=device) for m in train_mask}
    return batch_x, observed_t, train_mask_t

epochs = 50  # increase to 200+ for paper-level runs
for ep in range(1, epochs+1):
    model.train()
    total = total_rec = total_kl = 0.0
    for batch in loader:
        bx, obs_t, mask_t = to_tensor_batch(batch)
        refined_xhat, mus, logvars = model(bx, mask_t)
        loss, rec, kl = omicsmind_loss(bx, refined_xhat, mus, logvars, obs_t, mask_t, only_masked=False)

        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item(); total_rec += rec.item(); total_kl += kl.item()

    if ep % 5 == 0 or ep == 1:
        print(f"Epoch {ep:03d} | loss={total/len(loader):.4f} rec={total_rec/len(loader):.4f} kl={total_kl/len(loader):.4f}")


Epoch 001 | loss=1.8444 rec=1.8258 kl=18.5310
Epoch 005 | loss=11552949879648.0723 rec=1.8040 kl=11552949342775410.0000
Epoch 010 | loss=1.8004 rec=1.7934 kl=7.0585
Epoch 015 | loss=1.7489 rec=1.7358 kl=13.1416
Epoch 020 | loss=1.7253 rec=1.7159 kl=9.3492
Epoch 025 | loss=1.7282 rec=1.7188 kl=9.4196
Epoch 030 | loss=1.7354 rec=1.7192 kl=16.1835
Epoch 035 | loss=3116061246304.0225 rec=1.7369 kl=3116061163299272.5000
Epoch 040 | loss=1.7753 rec=1.7661 kl=9.1816
Epoch 045 | loss=1.7636 rec=1.7489 kl=14.7524
Epoch 050 | loss=6486163.3406 rec=1.7037 kl=6486161309.9594


## 9. Full imputation and CSV export


In [11]:
@torch.no_grad()
def full_impute(model, X_dict):
    model.eval()
    n = next(iter(X_dict.values())).shape[0]
    batch_x = {m: torch.tensor(np.nan_to_num(X_dict[m], nan=0.0), device=device) for m in X_dict}
    train_mask = {m: torch.ones(n, device=device) for m in X_dict}
    refined_xhat, _, _ = model(batch_x, train_mask)
    return {m: refined_xhat[m].cpu().numpy() for m in X_dict}

imputed_all = full_impute(model, X)

out_dir = "imputed_outputs"
os.makedirs(out_dir, exist_ok=True)
for m in imputed_all:
    df_out = pd.DataFrame(imputed_all[m], index=all_samples, columns=modalities_aligned[m].columns)
    df_out.to_csv(os.path.join(out_dir, f"{m}_imputed.csv"))
print("Saved to", out_dir, "files:", os.listdir(out_dir))


Saved to imputed_outputs files: ['meth_imputed.csv', 'rna_imputed.csv', 'rppa_imputed.csv', 'scna_imputed.csv']
